# SightPath — Exploration Notebook

This notebook walks through the project step by step.
Good for understanding what each module does and for checking intermediate results.

Run cells top to bottom.

In [ ]:
import sys, os
sys.path.insert(0, '..')   # so we can import from src/

import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

print('imports ok')

## 1. Build the graph

In [ ]:
from src.graph_builder import get_graph

# use_synthetic=True means no internet needed
G = get_graph(use_synthetic=True)
print(f'Nodes: {G.number_of_nodes()}')
print(f'Edges: {G.number_of_edges()}')

# peek at a few nodes
for n, d in list(G.nodes(data=True))[:3]:
    print(f'  node {n}: {d}')

## 2. Score edges with multi-objective features

In [ ]:
from src.feature_scorer import score_graph, normalise_graph_weights

G = score_graph(G, seed=42)
G = normalise_graph_weights(G)

# check a few edges
for u, v, k, d in list(G.edges(keys=True, data=True))[:3]:
    print(f'  ({u} -> {v}): dist={d["distance_m"]:.1f}m, '
          f'safety={d["safety_score"]:.3f}, '
          f'access={d["accessibility"]:.3f}, '
          f'comp={d["complexity"]:.3f}')

## 3. Distribution of edge features

In [ ]:
features = ['safety_score', 'accessibility', 'complexity', 'distance_m']
data = {f: [] for f in features}

for u, v, d in G.edges(data=True):
    for f in features:
        if f in d:
            data[f].append(d[f])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colours = ['#22c55e', '#3b82f6', '#ef4444', '#f59e0b']

for ax, feat, col in zip(axes, features, colours):
    ax.hist(data[feat], bins=25, color=col, alpha=0.8, edgecolor='white')
    ax.set_title(feat, fontsize=11)
    ax.set_xlabel('value')
    ax.set_ylabel('count')

plt.suptitle('Edge Feature Distributions', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('saved')

## 4. Pick source and target, run all algorithms

In [ ]:
import random

# pick endpoints with some path between them
nodes = list(G.nodes())
rng = random.Random(42)
for _ in range(100):
    src, tgt = rng.sample(nodes, 2)
    try:
        p = nx.shortest_path(G, src, tgt)
        if len(p) >= 6:
            source, target = src, tgt
            break
    except:
        pass

print(f'Source: {source}  |  Target: {target}')

In [ ]:
from src.algorithms import dijkstra, astar, nsga2_optimizer as nsga2

print('Running Dijkstra ...')
dijk = dijkstra.run(G, source, target, weight='distance_m')
print(f'  path length: {len(dijk["path"])} nodes, dist={dijk["distance_m"]}m')

print('Running A* ...')
astar_res = astar.run(G, source, target)
print(f'  path length: {len(astar_res["path"])} nodes, dist={astar_res["distance_m"]}m')

In [ ]:
print('Running NSGA-II (this takes ~30s) ...')
nsga2_res = nsga2.run(G, source, target, pop_size=50, n_generations=60, seed=42)
pf = nsga2_res['pareto_front']
print(f'Pareto front size: {len(pf)}')
print(f'Time: {nsga2_res["time_ms"]:.1f} ms')

## 5. Compare results

In [ ]:
from src.evaluator import build_comparison_table, print_comparison, hypervolume, spread

dijk_all = dijkstra.run_all_objectives(G, source, target)
df = build_comparison_table(dijk_all, astar_res, nsga2_res)
print_comparison(df, nsga2_res)

if pf:
    print(f'Hypervolume: {hypervolume(pf)}')
    print(f'Spread:      {spread(pf)}')

## 6. Visualise

In [ ]:
from src.visualizer import plot_routes, plot_pareto_front, plot_metrics_bar

routes = {
    'dijkstra': dijk_all.get('distance_m', {}).get('path', []),
    'astar':    astar_res.get('path', []),
}
if pf:
    best = nsga2.best_compromise(pf)
    if best:
        routes['nsga2'] = best.path

plot_routes(G, routes, source, target, '../outputs/notebook_routes.png')
from IPython.display import Image
Image('../outputs/notebook_routes.png')

In [ ]:
if pf:
    best_ind = nsga2.best_compromise(pf)
    plot_pareto_front(pf, highlight=best_ind, save_path='../outputs/notebook_pareto.png')
    Image('../outputs/notebook_pareto.png')

In [ ]:
if not df.empty:
    plot_metrics_bar(df, save_path='../outputs/notebook_bar.png')
    Image('../outputs/notebook_bar.png')